# Chapter 42: deep-learning mechanisms and a constructed comparison

Freshly check gradients and CNN/RNN/LSTM/attention operators, then fit Ridge and a 4→8→1 MLP on a constructed 360-day, 12-asset, four-feature panel. Compare the training-mean baseline, validation-selected Ridge and early-stopped MLP with shared prediction-to-ledger rules. The Notebook trains small teaching models and writes five new figures; it does not run the complete C01 contamination experiment or C02 fee grid/bisection.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch42_deep_learning_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
import matplotlib.pyplot as plt

from quant_course.deep_learning import (
    forward,
    loss_and_grad,
    predict,
    fit_mlp,
    cnn_cross_correlation,
    rnn_states,
    lstm_step,
    causal_attention,
    sinusoidal_positional_encoding,
    generate_ch42_panel_data,
    split_ch42_panel,
)
from quant_course.ml_pipeline import backtest_predictions

print("Environment successfully initialized. Core functions imported.")



## 1. MLP gradient check

For a 2→2→1 tanh network with row batches, $Z_1=XW_1+b_1$, $\hat Y=\tanh(Z_1)W_2+b_2$. Loss is $\|\hat Y-Y\|_F^2/(B D_{out})+\lambda(\|W_1\|_F^2+\|W_2\|_F^2)$; biases are not penalized. Compare every analytic parameter gradient with central differences at epsilon=10⁻⁶. Floating tolerance, not bit identity, is the check.

In [ ]:
rng = np.random.default_rng(4200)
B, D_in, D_hidden, D_out = 3, 2, 2, 1
X_micro = rng.normal(size=(B, D_in))
y_micro = rng.normal(size=(B, D_out))

W1 = rng.normal(size=(D_in, D_hidden))
b1 = rng.normal(size=(1, D_hidden))
W2 = rng.normal(size=(D_hidden, D_out))
b2 = rng.normal(size=(1, D_out))
l2_reg = 0.01

loss, dW1, db1, dW2, db2 = loss_and_grad(X_micro, y_micro, W1, b1, W2, b2, l2_reg=l2_reg)

print(f"Total Loss (MSE + L2): {loss:.6f}")

# 中心差分对比
eps = 1e-6
param_dict = {"W1": (W1, dW1), "b1": (b1, db1), "W2": (W2, dW2), "b2": (b2, db2)}
grad_check_results = []

for name, (param, analytic_g) in param_dict.items():
    num_g = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        orig = param[idx]
        param[idx] = orig + eps
        l_p, _, _, _, _ = loss_and_grad(X_micro, y_micro, W1, b1, W2, b2, l2_reg=l2_reg)
        param[idx] = orig - eps
        l_m, _, _, _, _ = loss_and_grad(X_micro, y_micro, W1, b1, W2, b2, l2_reg=l2_reg)
        param[idx] = orig
        num_g[idx] = (l_p - l_m) / (2.0 * eps)
        it.iternext()
    max_err = np.max(np.abs(analytic_g - num_g))
    grad_check_results.append({"Parameter": name, "Max Abs Error": f"{max_err:.2e}", "Status": "PASS" if max_err < 1e-7 else "FAIL"})

df_grad_check = pd.DataFrame(grad_check_results)
display(df_grad_check)


## 2. Architecture micro-examples

Valid cross-correlation of [1,2,3,4] with [1,−1] gives [−1,−1,−1]. The recurrence h_t=0.5h_(t−1)+x_t gives [1,2.5,1.25], with gradient factors 0.5^k. Inspect one LSTM step and a three-token positional encoding/attention calculation. Perturb the future token's Q, K and V and check that the earlier two outputs stay fixed under the strict causal mask. This local check does not repair upstream feature leakage.

In [ ]:
# 1. CNN Valid Cross-Correlation
x_cnn = [1.0, 2.0, 3.0, 4.0]
w_cnn = [1.0, -1.0]
cnn_out = cnn_cross_correlation(x_cnn, w_cnn)
print(f"CNN Cross-Correlation output of [1,2,3,4] with [1,-1]: {cnn_out}")
assert np.allclose(cnn_out, [-1.0, -1.0, -1.0])

# 2. Simple RNN Recurrence
x_rnn = [1.0, 2.0, 0.0]
rnn_out = rnn_states(x_rnn, w_h=0.5, w_x=1.0, b=0.0)
print(f"RNN states for x=[1, 2, 0]: {rnn_out}")
assert np.allclose(rnn_out, [1.0, 2.5, 1.25])
print("Gradient factor across k=3 steps: (0.5)^3 =", 0.5**3)

# 3. Modern LSTM Step
x_t = np.array([0.5])
h_prev = np.array([0.1])
c_prev = np.array([0.2])
W_lstm = {k: np.array([[1.0]]) for k in ("f", "i", "c", "o")}
U_lstm = {k: np.array([[0.5]]) for k in ("f", "i", "c", "o")}
b_lstm = {k: np.array([0.0]) for k in ("f", "i", "c", "o")}
lstm_res = lstm_step(x_t, h_prev, c_prev, W_lstm, U_lstm, b_lstm)
print("LSTM Step Output:")
for k, v in lstm_res.items():
    print(f"  {k}: {np.round(v, 4)}")

# 4. Transformer Causal Attention & Perturbation Invariance Test
Q = np.array([[1.,0.],[0.,1.],[1.,1.]])
K = np.array([[1.,1.],[1.,0.],[0.,1.]])
V = np.array([[1.,2.],[3.,4.],[5.,6.]])
print("Position encoding (T=3,d=4):", sinusoidal_positional_encoding(3,4))

att_weights, att_out = causal_attention(Q, K, V, mask=True)
print("\nCausal Attention Weights (Masked):\n", np.round(att_weights, 4))
print("Attention output:", np.round(att_out,4))

# 扰动未来 token 2
Q_pert, K_pert, V_pert = Q.copy(), K.copy(), V.copy()
Q_pert[2] += 10.0
K_pert[2] -= 5.0
V_pert[2] += 8.0
_, att_out_pert = causal_attention(Q_pert, K_pert, V_pert, mask=True)

diff_history = np.max(np.abs(att_out[:2] - att_out_pert[:2]))
print(f"Max difference on earlier tokens (0 and 1) after perturbing future token 2: {diff_history:.2e}")
assert diff_history == 0.0, "Leakage detected!"


## 3. Panel and strict boundary purging

Use 360 UTC weekdays, 12 assets and four features with constructed weak nonlinear signal plus noise SD 0.01. A signal at D_k closes trades at D_(k+1) closes and earns the return through D_(k+2). Strict maturity before boundaries D_180 and D_240 excludes signal dates 178,179,238,239. Train/validation/test have 178/58/118 mature dates, not that many independent observations.

In [ ]:
x, raw_return, close, dates, assets = generate_ch42_panel_data(seed=4201)
splits = split_ch42_panel(x, raw_return, dates, assets)

print(f"Panel Total Dates: {len(dates)}, Assets: {len(assets)}, Features: {x.shape[2]}")
print(f"Train mature dates: {len(splits.train_dates)} ({splits.train_dates[0]} to {splits.train_dates[-1]})")
print(f"Validation mature dates: {len(splits.val_dates)} ({splits.val_dates[0]} to {splits.val_dates[-1]})")
print(f"Test mature dates: {len(splits.test_dates)} ({splits.test_dates[0]} to {splits.test_dates[-1]})")
print("Purged boundary signal dates (preventing lookahead): [178, 179, 238, 239]")


## 4. Three models under one split

Estimate feature and target scaling on training only. The mean baseline has one parameter; Ridge selects alpha in {0.1,1,10} on validation data. The NumPy MLP has 49 parameters, monitors validation MSE and restores the best epoch. Multiple epochs and six ledger calls cost more than the preceding micro-examples.

In [ ]:
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

# 1. 特征与标签标准化（仅使用训练集均值和标准差）
mu_X = np.mean(splits.X_train, axis=0)
sigma_X = np.std(splits.X_train, axis=0)
mu_y = float(np.mean(splits.y_train))
sigma_y = float(np.std(splits.y_train))

X_train_s = (splits.X_train - mu_X) / sigma_X
y_train_s = (splits.y_train - mu_y) / sigma_y
X_val_s = (splits.X_val - mu_X) / sigma_X
y_val_s = (splits.y_val - mu_y) / sigma_y
X_test_s = (splits.X_test - mu_X) / sigma_X

# Model 1: Mean (1 parameter)
pred_mean = np.full_like(splits.y_test, mu_y)
mse_mean = float(np.mean((splits.y_test - pred_mean)**2))

# Model 2: Ridge
best_alpha, best_val_mse = None, float("inf")
for alpha in [0.1, 1.0, 10.0]:
    r = Ridge(alpha=alpha).fit(X_train_s, splits.y_train.ravel())
    v_mse = float(np.mean((splits.y_val.ravel() - r.predict(X_val_s))**2))
    if v_mse < best_val_mse:
        best_val_mse, best_alpha = v_mse, alpha

ridge_model = Ridge(alpha=best_alpha).fit(X_train_s, splits.y_train.ravel())
pred_ridge = ridge_model.predict(X_test_s).reshape(-1, 1)
mse_ridge = float(np.mean((splits.y_test - pred_ridge)**2))
r2_ridge = 1.0 - mse_ridge / mse_mean

# Model 3: MLP (4->8->1)
mlp_res = fit_mlp(
    X_train=X_train_s, y_train=y_train_s,
    X_val=X_val_s, y_val=y_val_s,
    hidden_dim=8, lr=0.01, batch_size=32, l2_reg=0.001,
    max_epochs=200, patience=15, seed_init=4202, seed_shuffle=4203
)
pred_mlp_s = predict(X_test_s, mlp_res.W1, mlp_res.b1, mlp_res.W2, mlp_res.b2)
pred_mlp = pred_mlp_s * sigma_y + mu_y
mse_mlp = float(np.mean((splits.y_test - pred_mlp)**2))
r2_mlp = 1.0 - mse_mlp / mse_mean

print(f"MLP Early stopping stopped at epoch {mlp_res.stopped_epoch}, restored best epoch {mlp_res.best_epoch}.")


## 5. Prediction and ledger evaluation

Demean predictions within each test date to form signed dollar-neutral weights with gross exposure 1; constant predictions imply no position. Compare MSE, R² and daily cross-sectional Spearman IC. Run each model at zero and 10 bp costs through the common ledger, with the same execution lag.

In [ ]:
# Daily Spearman IC
n_test_dates = len(splits.test_dates)
N = len(assets)
y_test_2d = splits.y_test.reshape(n_test_dates, N)
pred_ridge_2d = pred_ridge.reshape(n_test_dates, N)
pred_mlp_2d = pred_mlp.reshape(n_test_dates, N)

ic_ridge = [spearmanr(pred_ridge_2d[t], y_test_2d[t])[0] for t in range(n_test_dates)]
ic_mlp = [spearmanr(pred_mlp_2d[t], y_test_2d[t])[0] for t in range(n_test_dates)]

# Backtests
idx_test = pd.MultiIndex.from_product([splits.test_dates, assets], names=["signal_time", "asset"])
bt_mean_g = backtest_predictions(close, pd.Series(pred_mean.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.0)
bt_mean_n = backtest_predictions(close, pd.Series(pred_mean.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.001)

bt_ridge_g = backtest_predictions(close, pd.Series(pred_ridge.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.0)
bt_ridge_n = backtest_predictions(close, pd.Series(pred_ridge.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.001)

bt_mlp_g = backtest_predictions(close, pd.Series(pred_mlp.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.0)
bt_mlp_n = backtest_predictions(close, pd.Series(pred_mlp.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.001)

# Summary DataFrame
summary_table = pd.DataFrame({
    "Parameters": [1, 5, 49],
    "Test MSE": [f"{mse_mean:.8f}", f"{mse_ridge:.8f}", f"{mse_mlp:.8f}"],
    "Test R^2": ["0.000000", f"{r2_ridge:.6f}", f"{r2_mlp:.6f}"],
    "Daily Spearman IC": ["NaN (Constant)", f"{np.mean(ic_ridge):.4f}", f"{np.mean(ic_mlp):.4f}"],
    "Gross Sharpe": ["NaN", f"{bt_ridge_g['metrics']['sharpe']:.3f}", f"{bt_mlp_g['metrics']['sharpe']:.3f}"],
    "Net Sharpe (10 bps)": ["NaN", f"{bt_ridge_n['metrics']['sharpe']:.3f}", f"{bt_mlp_n['metrics']['sharpe']:.3f}"],
    "Annual Turnover": [0.0, f"{bt_ridge_n['metrics']['turnover_ann']:.1f}", f"{bt_mlp_n['metrics']['turnover_ann']:.1f}"],
}, index=["Global Training Mean", "Ridge Regression", "NumPy MLP (4->8->1)"])

display(summary_table)


## 6. Five fresh figures

Save training/validation losses, prediction-versus-target scatter, gross/net wealth, daily IC and the causal-attention heatmap from this run's arrays. They do not read the old chapter figures; the three-token heatmap is not the chapter's six-token example.

In [ ]:
# Use this cold run's computed arrays and fitted teaching model, not old saved figures.
def save_display(fig, filename):
    fig.tight_layout()
    fig.savefig(RUN_DIR / filename, dpi=150)
    display(Image(filename=str(RUN_DIR / filename)))
    plt.close(fig)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(mlp_res.train_loss_history, label="train")
ax.plot(mlp_res.val_loss_history, label="validation")
ax.axvline(mlp_res.best_epoch, linestyle="--", label="best epoch")
ax.set(xlabel="Epoch", ylabel="Scaled MSE", title="CONSTRUCTED MLP loss")
ax.legend(); save_display(fig, "loss_curve.png")

fig, axes = plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax, name, pred in zip(axes,["Ridge","MLP"],[pred_ridge,pred_mlp]):
    ax.scatter(pred.ravel(),splits.y_test.ravel(),s=3,alpha=.3)
    ax.set(xlabel="Predicted simple return",ylabel="Realized simple return",title="CONSTRUCTED " + name)
save_display(fig,"predictions_vs_targets.png")

fig, axes = plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax, title, results in zip(axes,["Gross","Net 10 bp"],[(bt_mean_g,bt_ridge_g,bt_mlp_g),(bt_mean_n,bt_ridge_n,bt_mlp_n)]):
    for name, result in zip(["Mean","Ridge","MLP"], results):
        nav=result["ledger"]["nav"]
        ax.plot(nav.index,nav/nav.iloc[0],label=name)
    ax.set(xlabel="Date",ylabel="Wealth (initial 1)",title="CONSTRUCTED " + title)
    ax.legend(); ax.tick_params(axis="x",rotation=30)
save_display(fig,"cumulative_returns.png")

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(splits.test_dates,ic_ridge,label="Ridge")
ax.plot(splits.test_dates,ic_mlp,label="MLP")
ax.set(xlabel="Test date",ylabel="Daily Spearman IC",title="CONSTRUCTED daily cross-sectional IC")
ax.legend(); ax.tick_params(axis="x",rotation=30)
save_display(fig,"daily_ic_timeseries.png")

fig, ax = plt.subplots(figsize=(5,4))
im=ax.imshow(att_weights,vmin=0,vmax=1)
ax.set(xlabel="Key token",ylabel="Query token",title="CONSTRUCTED causal attention")
fig.colorbar(im,ax=ax); save_display(fig,"causal_attention_weights.png")


## 7. Interpretation

The MLP's 49 parameters face 178 training dates (2,136 panel rows, not independent samples). A weak test R² against the mean and heavy turnover costs describe this constructed design, not a universal rejection of deep learning. Strict label boundaries and train-only preprocessing are distinct protections; a causal attention mask cannot remove leakage already embedded in features. Historical quoted Sharpe/turnover ranges should not replace the values printed by this actual run. No market alpha is established.